In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.cluster import KMeans

df = pd.read_csv("data.csv", )
pd.set_option('display.max_columns', None)

# Set display option to show float values in standard notation
pd.set_option('display.float_format', lambda x: '%.2f' % x)

display(df.head())

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Id,Customer Segment,Customer State,Customer Zipcode,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,Order_Date,Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Name,Product Price,Product Status,Shipping_Date,Shipping Mode,Post_Order_Action,Transportation_Cost,Lead_Time,Supplier_ID
0,DEBIT,3,4,91.25,314.64,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,20755,Consumer,PR,725.00,Fitness,18.25,-66.04,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.11,0.04,180517,327.75,0.29,1,327.75,314.64,91.25,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,NaN,23.64,6,sup001
1,TRANSFER,5,4,-249.09,311.36,Late delivery,1,Sporting Goods,Caguas,Puerto Rico,19492,Consumer,PR,725.00,Fitness,18.28,-66.04,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.39,0.05,179254,327.75,-0.80,1,327.75,311.36,-249.09,South Asia,Rajastán,PENDING,NaN,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,NaN,18.31,10,sup004
2,CASH,4,4,-247.78,309.72,Shipping on time,0,Sporting Goods,San Jose,EE. UU.,19491,Consumer,CA,95125.00,Fitness,37.29,-121.88,Pacific Asia,Bikaner,India,19491,2018-01-13 12:06:00,75938,1360,18.03,0.06,179253,327.75,-0.80,1,327.75,309.72,-247.78,South Asia,Rajastán,CLOSED,NaN,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,NaN,21.52,8,sup004
3,DEBIT,3,4,22.86,304.81,Advance shipping,0,Sporting Goods,Los Angeles,EE. UU.,19490,Home Office,CA,90027.00,Fitness,34.13,-118.29,Pacific Asia,Townsville,Australia,19490,2018-01-13 11:45:00,75937,1360,22.94,0.07,179252,327.75,0.08,1,327.75,304.81,22.86,Oceania,Queensland,COMPLETE,NaN,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,NaN,21.39,6,sup002
4,PAYMENT,2,4,134.21,298.25,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,19489,Corporate,PR,725.00,Fitness,18.25,-66.04,Pacific Asia,Townsville,Australia,19489,2018-01-13 11:24:00,75936,1360,29.50,0.09,179251,327.75,0.45,1,327.75,298.25,134.21,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,NaN,26.17,4,sup001


In [2]:
total_orders=len(df)
num_clusters = total_orders // 25000

# K-Means Clustering
kmeans = KMeans(n_clusters=num_clusters, random_state=0).fit(df[['Latitude', 'Longitude']])
df['Cluster'] = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

# Output the cluster centers
display(cluster_centers)


array([[  37.40917551,  -85.1291085 ],
       [  18.29753792,  -66.23719134],
       [  35.75844506, -118.25120731],
       [  40.24005781,  -75.04186386],
       [  32.82415627,  -99.33404607],
       [  29.10384478,   86.10965761],
       [  21.35195027, -157.88115923]])

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans

# Assuming df is your existing DataFrame with 'Latitude' and 'Longitude' columns
total_orders = len(df)
num_clusters = total_orders // 20000

# K-Means Clustering
kmeans = KMeans(n_clusters=num_clusters, random_state=0).fit(df[['Latitude', 'Longitude']])
df['Cluster'] = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

# Convert cluster centers to DataFrame
cluster_centers_df = pd.DataFrame(cluster_centers, columns=['Latitude', 'Longitude'])
country_count_df = pd.read_csv("geo_location.csv")

# Create choropleth map for country counts
fig = px.choropleth(
    country_count_df,
    locations="English_Name",
    locationmode="country names",
    color="count",
    hover_name="English_Name",
    color_continuous_scale=px.colors.sequential.Plasma,
    title="Suggested Warehouse Locations"
)

# Add cluster centers as a scatter plot
fig.add_trace(go.Scattergeo(
    lon=cluster_centers_df['Longitude'],
    lat=cluster_centers_df['Latitude'],
    text=['Cluster Center'] * len(cluster_centers_df),
    mode='markers',
    marker=dict(
        size=10,
        color='red',
        symbol='cross'
    ),
    name='Cluster Centers'
))

# Updating the layout to set the size
fig.update_layout(
    width=1400,
    height=800,
    geo=dict(
        showland=True,
        landcolor="lightgray",
    )
)

# Show the plot
fig.show()


In [4]:
from sklearn.cluster import KMeans

# Assuming df is your existing DataFrame with 'Latitude' and 'Longitude' and 'Cluster' columns

# Determine the number of distribution centers per warehouse
distribution_centers_per_warehouse = 3  # Example number, adjust based on your needs

# DataFrame to store distribution center locations
distribution_centers_df = pd.DataFrame(columns=['Warehouse_Cluster', 'Latitude', 'Longitude'])

for warehouse_cluster in df['Cluster'].unique():
    # Filter data points belonging to the current warehouse cluster
    warehouse_cluster_data = df[df['Cluster'] == warehouse_cluster]
    
    # Apply K-Means clustering to determine distribution centers within this warehouse cluster
    kmeans_dc = KMeans(n_clusters=distribution_centers_per_warehouse, random_state=0)
    kmeans_dc.fit(warehouse_cluster_data[['Latitude', 'Longitude']])
    
    # Extract distribution center coordinates
    dc_centers = kmeans_dc.cluster_centers_
    dc_centers_df = pd.DataFrame(dc_centers, columns=['Latitude', 'Longitude'])
    dc_centers_df['Warehouse_Cluster'] = warehouse_cluster
    
    # Append to the main DataFrame
    distribution_centers_df = pd.concat([distribution_centers_df, dc_centers_df], ignore_index=True)

# Plot the distribution centers on the map
fig.add_trace(go.Scattergeo(
    lon=distribution_centers_df['Longitude'],
    lat=distribution_centers_df['Latitude'],
    text=['Distribution Center'] * len(distribution_centers_df),
    mode='markers',
    marker=dict(
        size=8,
        color='green',
        symbol='circle'
    ),
    name='Distribution Centers'
))

# Show the plot
fig.show()


C:\Users\Admin\AppData\Local\Temp\ipykernel_1816\1158326528.py:25: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



In [7]:
import pandas as pd
import plotly.graph_objects as go
from sklearn.cluster import KMeans

# Assuming df is your existing DataFrame with 'Latitude', 'Longitude', and 'Cluster' columns
total_orders = len(df)
num_clusters = total_orders // 25000

# K-Means Clustering for Warehouses
kmeans = KMeans(n_clusters=num_clusters, random_state=0).fit(df[['Latitude', 'Longitude']])
df['Cluster'] = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

# Convert cluster centers to DataFrame
cluster_centers_df = pd.DataFrame(cluster_centers, columns=['Latitude', 'Longitude'])
cluster_centers_df['Cluster'] = range(len(cluster_centers_df))

# DataFrame to store distribution center locations
distribution_centers_df = pd.DataFrame(columns=['Warehouse_Cluster', 'Latitude', 'Longitude'])

# Set the density threshold (one distribution center per 5000 customers)
density_threshold = 5000

for warehouse_cluster in df['Cluster'].unique():
    # Filter data points belonging to the current warehouse cluster
    warehouse_cluster_data = df[df['Cluster'] == warehouse_cluster]
    
    # Determine the number of distribution centers needed for this warehouse cluster
    num_distribution_centers = max(1, len(warehouse_cluster_data) // density_threshold)
    
    # Apply K-Means clustering to determine distribution centers within this warehouse cluster
    kmeans_dc = KMeans(n_clusters=num_distribution_centers, random_state=0)
    kmeans_dc.fit(warehouse_cluster_data[['Latitude', 'Longitude']])
    
    # Extract distribution center coordinates
    dc_centers = kmeans_dc.cluster_centers_
    dc_centers_df = pd.DataFrame(dc_centers, columns=['Latitude', 'Longitude'])
    dc_centers_df['Warehouse_Cluster'] = warehouse_cluster
    
    # Append to the main DataFrame
    distribution_centers_df = pd.concat([distribution_centers_df, dc_centers_df], ignore_index=True)

# Colors for different warehouse clusters
colors = px.colors.qualitative.Plotly

fig = go.Figure()

# Plot warehouses
for i, warehouse_cluster in enumerate(cluster_centers_df['Cluster'].unique()):
    cluster_color = colors[i % len(colors)]
    cluster_warehouse = cluster_centers_df[cluster_centers_df['Cluster'] == warehouse_cluster]
    fig.add_trace(go.Scattergeo(
        lon=cluster_warehouse['Longitude'],
        lat=cluster_warehouse['Latitude'],
        text=cluster_warehouse['Cluster'],
        marker=dict(
            size=10,
            color=cluster_color,
            symbol='x'
        ),
        name=f'Warehouse {warehouse_cluster}'
    ))

# Plot distribution centers
for i, warehouse_cluster in enumerate(cluster_centers_df['Cluster'].unique()):
    cluster_color = colors[i % len(colors)]
    cluster_dc = distribution_centers_df[distribution_centers_df['Warehouse_Cluster'] == warehouse_cluster]
    fig.add_trace(go.Scattergeo(
        lon=cluster_dc['Longitude'],
        lat=cluster_dc['Latitude'],
        text=cluster_dc['Warehouse_Cluster'],
        marker=dict(
            size=7,
            color=cluster_color,
            symbol='circle'
        ),
        name=f'Distribution Centers of Warehouse {warehouse_cluster}'
    ))

# Set the map's layout
fig.update_layout(
    title="Warehouse and Distribution Center Locations",
    geo=dict(
        scope='world',
        projection=go.layout.geo.Projection(type='equirectangular'),
        showland=True,
    ),
    width=1500,
    height=800
)

# Show the plot
fig.show()


C:\Users\Admin\AppData\Local\Temp\ipykernel_1816\4249349123.py:41: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



In [6]:
import pandas as pd
import plotly.graph_objects as go
from sklearn.cluster import KMeans

# Assuming df is your existing DataFrame with 'Latitude', 'Longitude', and 'Cluster' columns
total_orders = len(df)
num_clusters = total_orders // 25000

# K-Means Clustering for Warehouses
kmeans_warehouses = KMeans(n_clusters=num_clusters, random_state=0).fit(df[['Latitude', 'Longitude']])
df['Warehouse_Cluster'] = kmeans_warehouses.labels_
warehouse_centers = kmeans_warehouses.cluster_centers_

# Convert warehouse centers to DataFrame
warehouse_centers_df = pd.DataFrame(warehouse_centers, columns=['Latitude', 'Longitude'])
warehouse_centers_df['Cluster'] = range(len(warehouse_centers_df))

# Determine the number of distribution centers needed (one per 5000 customers)
num_distribution_centers = total_orders // 5000

# K-Means Clustering for Distribution Centers
kmeans_dcs = KMeans(n_clusters=num_distribution_centers, random_state=0).fit(df[['Latitude', 'Longitude']])
df['DC_Cluster'] = kmeans_dcs.labels_
dc_centers = kmeans_dcs.cluster_centers_

# Convert distribution center centers to DataFrame
dc_centers_df = pd.DataFrame(dc_centers, columns=['Latitude', 'Longitude'])
dc_centers_df['Cluster'] = range(len(dc_centers_df))

fig = go.Figure()

# Plot warehouses in red with cross symbols
fig.add_trace(go.Scattergeo(
    lon=warehouse_centers_df['Longitude'],
    lat=warehouse_centers_df['Latitude'],
    text=warehouse_centers_df['Cluster'],
    marker=dict(
        size=10,
        color='red',
        symbol='x'
    ),
    name='Warehouses'
))

# Plot distribution centers in blue with circle symbols
fig.add_trace(go.Scattergeo(
    lon=dc_centers_df['Longitude'],
    lat=dc_centers_df['Latitude'],
    text=dc_centers_df['Cluster'],
    marker=dict(
        size=7,
        color='blue',
        symbol='circle'
    ),
    name='Distribution Centers'
))

# Set the map's layout
fig.update_layout(
    title="Warehouse and Distribution Center Locations",
    geo=dict(
        scope='world',
        projection=go.layout.geo.Projection(type='equirectangular'),
        showland=True,
    ),
    width=1200,
    height=800
)

# Show the plot
fig.show()
